### API keys for ECMWF
1da9b1baec5fa2a6c9b847be87b7efe0 (valid until Aug. 8, 2024, 1:53 a.m.)

### Email
jaguir26@ucsc.edu

### Content of $HOME/.ecmwfapirc

{
    
    "url"   : "https://api.ecmwf.int/v1",

    "key"   : "1da9b1baec5fa2a6c9b847be87b7efe0",
    
    "email" : "jaguir26@ucsc.edu"
}

### API key for Copernicus
Data download requests can be sent programatically via our API. However, a user ID and API key must be sent using HTTP basic authentication.

{
    
    "UID"      : "248017",
    
    "API Key"  : "1535164c-d38c-48bc-ba60-0c911710e4a8"
}

In [ ]:
import cdsapi
import os

def retrieve_glofas_data(
    system_version=['operational', 'version_2_1'], 
    hydrological_model=['htessel_lisflood', 'lisflood'], 
    product_type=['control_forecast', 'ensemble_perturbed_forecasts'], 
    variable='river_discharge_in_the_last_24_hours',
    year='2023',
    month='10',
    day='23',
    leadtime_hour=[
                '24', '48', '72',
                '96', '120', '144',
                '168', '192', '216',
                '240', '264', '288',
                '312', '336', '360',
                '384', '408', '432',
                '456', '480', '504',
                '528', '552', '576',
                '600', '624', '648',
                '672', '696', '720',
            ],
    target_folder='/home/antonio/UCSC/Research/GloFAS_data/', 
    area=[37.1, -122, 37, -121.9]  # Bounding box around San Lorenzo Station at Big Trees
):
    # Create a file name based on the retrieval parameters
    file_name = f"{system_version}_{hydrological_model}_{'_'.join(product_type)}_{variable}_{year}{month}{day}_{leadtime_hour}"
    
    # Include the 'area' in the file name if provided, otherwise add a distinguishing string
    if area:
        file_name += f"_area_{area[0]}_{area[1]}_{area[2]}_{area[3]}.grib"
    else:
        file_name += "_no_area.grib"
    
    output_file = os.path.join(target_folder, file_name)

    # Check if the file already exists in the target folder
    if not os.path.exists(output_file):
        # Initialize the CDS API client
        c = cdsapi.Client()

        # Define the retrieval parameters
        retrieval_params = {
            'system_version': system_version,
            'hydrological_model': hydrological_model,
            'product_type': product_type,
            'variable': variable,
            'year': year,
            'month': month,
            'day': day,
            'leadtime_hour': leadtime_hour,
            'format': 'grib',
        }

        # Include the 'area' parameter if provided
        if area:
            retrieval_params['area'] = area

        # Perform the retrieval
        c.retrieve('cems-glofas-forecast', retrieval_params, output_file)

        print("Retrieval completed. Output file saved to:", output_file)
    else:
        print("File already exists:", output_file)



In [ ]:
# CA
retrieve_glofas_data(
    system_version='operational',
    hydrological_model='lisflood',
    product_type=['control_forecast', 'ensemble_perturbed_forecasts'],
    variable='river_discharge_in_the_last_24_hours',
    year='2023',
    month='09',
    day='27',
    leadtime_hour='24',
    target_folder='/home/antonio/UCSC/Research/GloFAS_data/',
    area=[42, -125, 32, -114]
)

# Santa Cruz Area
retrieve_glofas_data(
    system_version='operational',
    hydrological_model='lisflood',
    product_type=['control_forecast', 'ensemble_perturbed_forecasts'],
    variable='river_discharge_in_the_last_24_hours',
    year='2023',
    month='09',
    day='27',
    leadtime_hour='24',
    target_folder='/home/antonio/UCSC/Research/GloFAS_data/',
    area=[37.6, -123, 36.4, -121]
)

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# Load and merge data from a GRIB file
def merge_data_types_in_grib(file_path):
    ds_cf = xr.open_dataset(file_path, engine="cfgrib", backend_kwargs={"filter_by_keys": {"dataType": "cf"}})
    ds_cf = ds_cf.rename_vars({'dis24': 'dis24_cf'})
    
    ds_pf = xr.open_dataset(file_path, engine="cfgrib", backend_kwargs={"filter_by_keys": {"dataType": "pf"}})
    ds_pf = ds_pf.rename_vars({'dis24': 'dis24_pf'})
    
    return xr.merge([ds_cf, ds_pf])

# Plot the transformed mean discharge grid
def plot_transformed_mean_grid(data):
    transformed_data = np.log(data + 1)
    plt.figure(figsize=(12, 6))
    transformed_data.plot(cmap='viridis', extend='both', cbar_kwargs={'label': 'Log-transformed Discharge ($\ln(m^3 \cdot s^{-1} + 1)$)'})
    plt.title('Log-transformed Mean Discharge in the Last 24 Hours')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.xlim([transformed_data.longitude.min(), transformed_data.longitude.max()])
    plt.ylim([transformed_data.latitude.min(), transformed_data.latitude.max()])
    plt.show()

# Plot each ensemble member in a matrix
def plot_ensemble_matrix(data):
    fig, axs = plt.subplots(nrows=7, ncols=8, figsize=(40, 28), sharex=True, sharey=True)
    axs[-1, -1].axis('off')  # Remove the last subplot
    image = None
    for i, ax in enumerate(axs.ravel()[:len(data.number)]):
        member_data = np.log(data.isel(number=i) + 1)
        image = ax.pcolormesh(member_data.longitude, member_data.latitude, member_data, cmap='viridis')
        ax.set_title(f'Member {i}', fontsize=10)
        ax.set_xlabel('')
        ax.set_ylabel('')
    cbar_ax = fig.add_axes([0.92, 0.12, 0.02, 0.75])
    fig.colorbar(image, cax=cbar_ax, label='Log-transformed Discharge ($\ln(m^3 \cdot s^{-1} + 1)$)')
    plt.tight_layout()
    plt.subplots_adjust(right=0.9)
    plt.show()



In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Load and merge data from a GRIB file
def merge_data_types_in_grib(file_path):
    ds_cf = xr.open_dataset(file_path, engine="cfgrib", backend_kwargs={"filter_by_keys": {"dataType": "cf"}})
    ds_cf = ds_cf.rename_vars({'dis24': 'dis24_cf'})
    
    ds_pf = xr.open_dataset(file_path, engine="cfgrib", backend_kwargs={"filter_by_keys": {"dataType": "pf"}})
    ds_pf = ds_pf.rename_vars({'dis24': 'dis24_pf'})
    
    return xr.merge([ds_cf, ds_pf])

# Plot the transformed mean discharge grid
def plot_transformed_mean_grid(data, vmin=0, vmax=6):
    transformed_data = np.log(data + 1)
    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    
    # Plot transformed data with specified color range
    transformed_data.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='viridis', extend='both', 
                          cbar_kwargs={'label': 'Log-transformed Discharge ($\ln(m^3 \cdot s^{-1} + 1)$)'}, 
                          vmin=vmin, vmax=vmax)
    
    # Add features
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.STATES)
    ax.plot(-122.0308, 36.9741, 'ro', markersize=3)  # Santa Cruz location
    plt.title('Log-transformed Mean Discharge in the Last 24 Hours')
    plt.show()

def plot_ensemble_matrix(data, vmin=0, vmax=6):
    fig, axs = plt.subplots(nrows=7, ncols=8, figsize=(40, 28), 
                            subplot_kw={'projection': ccrs.PlateCarree()}, sharex=True, sharey=True)
    axs[-1, -1].axis('off')  # Remove the last subplot
    images = []
    for i, ax in enumerate(axs.ravel()[:len(data.number)]):
        member_data = np.log(data.isel(number=i) + 1)
        
        # Plot transformed data with specified color range and store the returned image object
        img = member_data.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='viridis', add_colorbar=False, vmin=vmin, vmax=vmax)
        images.append(img)
        
        # Add features
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS)
        ax.add_feature(cfeature.STATES)
        ax.plot(-122.0308, 36.9741, 'ro', markersize=6)  # Santa Cruz location
        ax.set_title(f'Member {i}', fontsize=10)
        
        # Set bounds to match the area covered by the dis24 data
        ax.set_extent([data.longitude.min(), data.longitude.max(), data.latitude.min(), data.latitude.max()])
    
    # Common colorbar for all subplots using the first image as reference
    cbar_ax = fig.add_axes([0.92, 0.12, 0.02, 0.75])
    fig.colorbar(images[0], cax=cbar_ax, label='Log-transformed Discharge ($\ln(m^3 \cdot s^{-1} + 1)$)')
    plt.tight_layout()
    plt.subplots_adjust(right=0.9)
    plt.show()

# Plot the transformed mean discharge grid with non-zero values
def plot_non_zero_transformed_mean_grid(data, vmin=0, vmax=6):
    transformed_data = np.log(data + 1)
    transformed_data = transformed_data.where(transformed_data > 0)  # Mask zero values
    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    
    # Plot transformed data with specified color range
    transformed_data.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='viridis', extend='both', 
                          cbar_kwargs={'label': 'Log-transformed Discharge ($\ln(m^3 \cdot s^{-1} + 1)$)'}, 
                          vmin=vmin, vmax=vmax)
    
    # Add features
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.STATES)
    ax.plot(-122.0308, 36.9741, 'ro', markersize=3)  # Santa Cruz location
    plt.title('Log-transformed Mean Discharge in the Last 24 Hours (Non-zero values)')
    plt.show()

# Plot the ensemble matrix with non-zero values
def plot_non_zero_ensemble_matrix(data, vmin=0, vmax=6):
    fig, axs = plt.subplots(nrows=7, ncols=8, figsize=(40, 28), 
                            subplot_kw={'projection': ccrs.PlateCarree()}, sharex=True, sharey=True)
    axs[-1, -1].axis('off')  # Remove the last subplot
    images = []
    for i, ax in enumerate(axs.ravel()[:len(data.number)]):
        member_data = np.log(data.isel(number=i) + 1)
        member_data = member_data.where(member_data > 0)  # Mask zero values
        
        # Plot transformed data with specified color range and store the returned image object
        img = member_data.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='viridis', add_colorbar=False, vmin=vmin, vmax=vmax)
        images.append(img)
        
        # Add features
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS)
        ax.add_feature(cfeature.STATES)
        ax.plot(-122.0308, 36.9741, 'ro', markersize=6)  # Santa Cruz location
        ax.set_title(f'Member {i}', fontsize=10)
        
        # Set bounds to match the area covered by the dis24 data
        ax.set_extent([data.longitude.min(), data.longitude.max(), data.latitude.min(), data.latitude.max()])
    
    # Common colorbar for all subplots using the first image as reference
    cbar_ax = fig.add_axes([0.92, 0.12, 0.02, 0.75])
    fig.colorbar(images[0], cax=cbar_ax, label='Log-transformed Discharge ($\ln(m^3 \cdot s^{-1} + 1)$)')
    plt.tight_layout()
    plt.subplots_adjust(right=0.9)
    plt.show()

In [ ]:
# Main script execution
file_path_SC = '/home/antonio/UCSC/Research/GloFAS_data/operational_lisflood_control_forecast_ensemble_perturbed_forecasts_river_discharge_in_the_last_24_hours_20230919_720_area_37.6_-123_36.4_-121.grib'
file_path_CA = '/home/antonio/UCSC/Research/GloFAS_data/operational_lisflood_control_forecast_ensemble_perturbed_forecasts_river_discharge_in_the_last_24_hours_20230919_720_area_42_-125_32_-114.grib'

files = [file_path_CA, file_path_SC]

for file_path in files:
    merged_dataset = merge_data_types_in_grib(file_path)
    dis24_cf_expanded = merged_dataset['dis24_cf'].expand_dims(number=[0])
    all_forecasts = xr.concat([dis24_cf_expanded, merged_dataset['dis24_pf']], dim='number')
    mean_grid = all_forecasts.mean(dim='number')
    plot_transformed_mean_grid(mean_grid)
    plot_ensemble_matrix(all_forecasts)
    plot_non_zero_transformed_mean_grid(mean_grid)
    plot_non_zero_ensemble_matrix(all_forecasts)


### Big Trees San Lorenzo coordinates:
- Latitude: 37.0443931	
- Longitude: -122.072464 

In [ ]:
from datetime import date, timedelta

# Define start and end dates

start_date = date(2021, 1, 1)
end_date = date(2023, 10, 23)
delta = timedelta(days=1)

current_date = start_date

# Loop through each day from start_date to end_date
while current_date <= end_date:
    year = str(current_date.year)
    month = str(current_date.month).zfill(2)
    day = str(current_date.day).zfill(2)
    
    print(f"Retrieving data for {current_date}...")  # Debug line

    # Call your existing function
    retrieve_glofas_data(
        system_version=['operational', 'version_2_1'],
        hydrological_model=['htessel_lisflood', 'lisflood'],
        product_type=['control_forecast', 'ensemble_perturbed_forecasts'],
        variable='river_discharge_in_the_last_24_hours',
        year=year,
        month=month,
        day=day,
        leadtime_hour='24',
        target_folder='/home/antonio/UCSC/Research/GloFAS_data/',
        area=[37.1, -122, 37, -121.9]  # Bounding box around San Lorenzo Station at Big Trees
    )
    
    # Increment the current_date by one day
    current_date += delta




In [ ]:
# Clear the workspace
gc()

# Define the list of libraries
libraries_to_install <- c(
  "dataRetrieval",
  "dplyr"
)

# Function to check if a package is installed; if not, install it
install_if_missing <- function(package) {
  if (!requireNamespace(package, quietly = TRUE)) {
    install.packages(package, dependencies = TRUE)
  }
}

# Install and load the libraries
invisible(sapply(libraries_to_install, install_if_missing))
lapply(libraries_to_install, library, character.only = TRUE)

# Verify the libraries are loaded
search()

# Set working directory
setwd('/home/antonio/UCSC/Research')

# Read USGS data
site_code <- "11161000"
data_usgs_r <- readNWISdv(siteNumbers = site_code, parameterCd = "00060", statCd = "00003")

# Manipulate USGS data
San_Lorenzo_Daily_USGS_R <- data_usgs_r %>%
  mutate(timestamp = as.Date(Date),
         data0 = log(X_00060_00003 + 1)) %>%
  filter(timestamp > as.Date("1987-01-01"))


In [ ]:
# Define directory structure
main_folder <- '/home/antonio/UCSC/Research/USGS river data'
sub_folder <- 'San Lorenzo River at Big Trees'

# Check if main folder exists; if not, create it
if (!dir.exists(main_folder)) {
  dir.create(main_folder)
}

# Check if subfolder exists; if not, create it
full_sub_folder_path <- file.path(main_folder, sub_folder)
if (!dir.exists(full_sub_folder_path)) {
  dir.create(full_sub_folder_path)
}

# Save data
write.csv(San_Lorenzo_Daily_USGS_R, file.path(full_sub_folder_path, 'San_Lorenzo_Daily_USGS_R.csv'))


In [ ]:
usgs_file_path = '/home/antonio/UCSC/Research/USGS river data/San Lorenzo River at Big Trees/San_Lorenzo_Daily_USGS_R.csv'
usgs_data = pd.read_csv(usgs_file_path)
usgs_data['timestamp'] = pd.to_datetime(usgs_data['timestamp'])

In [ ]:
import os
import pygrib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_closest_point(lat_array, lon_array, target_lat, target_lon):
    # Compute the distance squared
    dist_sq = (lat_array - target_lat)**2 + (lon_array - target_lon)**2
    return np.unravel_index(np.argmin(dist_sq), dist_sq.shape)

# Your target latitude and longitude
target_lat = 37.0443931
target_lon = -122.072464

# Directory where the GloFAS data is stored
data_dir = '/home/antonio/UCSC/Research/GloFAS_data/'

# Check if the directory exists
if not os.path.exists(data_dir):
    print(f"The directory {data_dir} does not exist.")
else:
    # List GRIB files in the directory
    grib_files = [f for f in os.listdir(data_dir) if f.endswith('.grib')]

    if len(grib_files) == 0:
        print("No GRIB files found in the directory.")
    else:
        grib_files.sort()  # Sort the files (optional)

        # Dictionary to hold time series data for each ensemble
        ensemble_time_series = {i: [] for i in range(51)}  # Assuming 51 ensembles

        # Loop over all the GRIB files in the directory
        for file in grib_files:
            dis24_values = []  # Reset the list to store 'dis24' values for this file

            with pygrib.open(os.path.join(data_dir, file)) as grbs:
                for grb in grbs:
                    if 'dis24' in grb.shortName:
                        data = grb.values
                        lats, lons = grb.latlons()
                        closest_idx = find_closest_point(lats, lons, target_lat, target_lon)
                        dis24_values.append(data[closest_idx])

            # Append the dis24 values to the corresponding time series
            for i, val in enumerate(dis24_values):
                ensemble_time_series[i].append(val)

        # Convert lists to NumPy arrays for easier manipulation later
        for i in range(51):
            ensemble_time_series[i] = np.array(ensemble_time_series[i])

        print("GloFAS data loaded successfully.")


In [ ]:
# Read the USGS Data
usgs_file_path = '/home/antonio/UCSC/Research/USGS river data/San Lorenzo River at Big Trees/San_Lorenzo_Daily_USGS_R.csv'
usgs_data = pd.read_csv(usgs_file_path)
usgs_data['timestamp'] = pd.to_datetime(usgs_data['timestamp'])

print("USGS data loaded successfully.")


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pygrib  # Importing the pygrib library for reading GRIB files


# Assume GloFAS_data_start_date is the starting date of your GloFAS data series
# Format should be 'YYYY-MM-DD'
GloFAS_data_start_date = '2022-01-01'

# Create a Pandas DateTimeIndex starting from the GloFAS_data_start_date and having as many points as one of your ensemble series
glofas_time_index = pd.date_range(start=GloFAS_data_start_date, periods=len(ensemble_time_series[0]), freq='D')


# Initialize the plot
plt.figure(figsize=(20, 10))

# Plotting the ensemble time series
for ensemble, time_series in ensemble_time_series.items():
    transformed_time_series = np.log(time_series + 1)  # Add 1 and then take the natural log
    plt.plot(glofas_time_index, transformed_time_series, alpha=0.5)  # Using alpha for better visibility

# Plot USGS data as small points
plt.scatter(usgs_data['timestamp'], usgs_data['data0'], s=10, color='green', marker='o')

# Add titles and labels
plt.title('Time Series of GloFAS and USGS Data')
plt.xlabel('Time (Days)')
plt.ylabel('Data Value (log(x+1) for GloFAS Ensembles)')

# Show the plot
plt.show()




In [ ]:
import pandas as pd

# Assuming GloFAS data starts from '2021-01-01' and is daily
glofas_dates = pd.date_range(start='2022-01-01', periods=656, freq='D')

# Assign this date range to a DataFrame
glofas_df = pd.DataFrame({'timestamp': glofas_dates})

print(glofas_df.head())
usgs_df = pd.DataFrame({
    'timestamp': usgs_data['timestamp'],
    'data0': usgs_data['data0']
})

merged_df = pd.merge(usgs_df, glofas_df, how='inner', on='timestamp')
# Filter ensemble data based on common timestamps
filtered_ensemble_time_series = {key: value[merged_df.index] for key, value in ensemble_time_series.items()}

plt.figure(figsize=(15, 8))

# Plot ensemble time series
for ensemble, time_series in filtered_ensemble_time_series.items():
    transformed_time_series = np.log(time_series + 1)
    plt.plot(merged_df['timestamp'], transformed_time_series, color='blue', alpha=0.2)

# Plot USGS data
plt.scatter(merged_df['timestamp'], np.log(merged_df['data0'] + 1), s=10, color='green', marker='o')

# Titles and labels
plt.title('Ensemble and USGS Data Since 2021')
plt.xlabel('Timestamp')
plt.ylabel('Log-transformed Data')

plt.show()
